In [ ]:
import numpy as np
import pandas as pd
import scipy.spatial
from scipy import stats, interpolate
import matplotlib
import matplotlib.pyplot as plt

# Mismo modelo que el original: gaussiana multivariante condicional p(y|x).
from haloscope import ConditionalMultiVariateGaussian

In [ ]:
# ------------------------------- Configuración -------------------------------
# --- SIM: simulación DMO grande (fuente de verdad, con TODAS las propiedades) ---
SIM_BOXSIZE = 205.0          # tamaño de caja de SIM [Mpc/h]  <-- AJUSTA
SIM_NSIM    = 1              # nº de realizaciones/cajas de SIM a concatenar
SIM_PATH    = "Data/SIM/CV_"                 # prefijo (si hay varias cajas: CV_0, CV_1, ...)
SIM_HLIST   = "/hlists/hlist_1.00000.list"   # snapshot z=0 (formato Rockstar/consistent-trees)
SIM_HEADER  = 63                             # nº de filas de cabecera del hlist  <-- VERIFICA

# --- FASTPM: catálogo a enriquecer (SOLO posiciones + masas) ---
FASTPM_BOXSIZE = 1000.0      # tamaño de caja de FASTPM [Mpc/h]  <-- AJUSTA
FASTPM_PATH    = "Data/FASTPM/halos_z0.parquet"   # <-- formato/ruta reales

# --- Aprendizaje ---
# x (entrada): lo que FASTPM SÍ tiene. Se binea por masa y se condiciona en 'env'.
INPUT_FEATURES  = ["env"]                    # proxy de TidalForce, computable en FASTPM
OUTPUT_FEATURES = ["cv", "Spin", "ca", "ba"] # lo que queremos generar en FASTPM

ENV_RADIUS = 5.0             # radio [Mpc/h] para el proxy de entorno (densidad local)
CALIBRATE_MASS = True        # remapear masa FASTPM -> función de masa de SIM

In [ ]:
def bin_midpoints(edges):
    """Puntos medios de un vector de bordes de bins."""
    return (edges[1:] + edges[:-1]) / 2


def confidence_intervals(z, dx=1.0, dy=1.0):
    """Niveles de contorno que encierran 95%, 68% y 40% de la probabilidad.
    `z` es una PDF 2D normalizada (histograma / histograma.sum())."""
    n = 20
    t = np.linspace(0, z.max(), n)
    integral = ((z >= t[:, None, None]) * z).sum(axis=(1, 2)) * dx * dy
    f = interpolate.interp1d(integral, t)
    return np.append(f(np.array([0.95, 0.68, 0.40])), 1)


def load_hlists(path_prefix, header, columns, usecols, nsim, hlist):
    """Carga y concatena catálogos Rockstar/consistent-trees de `nsim` cajas."""
    frames = []
    for i in range(nsim):
        fname = f"{path_prefix}{i}{hlist}" if nsim > 1 else f"{path_prefix.rstrip('_')}{hlist}"
        frames.append(pd.read_table(
            fname, index_col=False, header=header, sep=r"\s+",
            names=columns, usecols=usecols, low_memory=False))
    return pd.concat(frames, ignore_index=True)

In [ ]:
def local_environment(pos, boxsize, radius=ENV_RADIUS, workers=5, log=True):
    """Proxy de entorno/marea a partir de SOLO posiciones.

    Densidad local = nº de halos vecinos dentro de `radius` (KD-tree periódico).
    Es el sustituto de `TidalForce`, computable tanto en SIM como en FASTPM.
    Debe calcularse de forma IDÉNTICA en ambas para que la feature sea consistente.

    `pos` : (N, 3) posiciones en Mpc/h.
    Devuelve un vector (N,) [log10(1 + nº de vecinos) si log=True].
    """
    tree = scipy.spatial.cKDTree(pos, boxsize=boxsize)
    counts = tree.query_ball_point(pos, r=radius, workers=workers, return_length=True)
    counts = counts - 1                       # excluir el propio halo
    return np.log10(1.0 + counts) if log else counts.astype(float)


def abundance_match_mass(mass_fastpm, mass_sim):
    """Recalibra la masa de FASTPM sobre la función de masa de SIM (matching monótono).

    Empareja por rango: el halo más masivo de FASTPM recibe la masa del más masivo
    de SIM, etc. (interpolación por percentiles). Corrige el sesgo de que las masas
    de FastPM son aproximadas y de definición distinta a la M200b de Rockstar.
    Devuelve la masa de FASTPM en la 'escala' de SIM.
    """
    q_fastpm = stats.rankdata(mass_fastpm) / (len(mass_fastpm) + 1)   # percentil de cada halo
    sim_sorted = np.sort(mass_sim)
    q_sim = (np.arange(len(sim_sorted)) + 1) / (len(sim_sorted) + 1)
    return np.interp(q_fastpm, q_sim, sim_sorted)

In [ ]:
SIM_COLS = ["id", "pid", "Rvir", "x", "y", "z", "Spin", "Rs_Klypin",
            "M200b", "ba", "ca", "ba500c", "ca500c", "Vpeak", "TidalForce"]
SIM_USECOLS = (1, 5, 11, 17, 18, 19, 26, 37, 39, 46, 47, 51, 52, 66, 81)  # <-- VERIFICA

data_sim = load_hlists(SIM_PATH, SIM_HEADER, SIM_COLS, SIM_USECOLS,
                       SIM_NSIM, SIM_HLIST)
data_sim.insert(9, "cv", data_sim.Rvir / data_sim.Rs_Klypin)

# Solo halos anfitriones y con masa/concentración físicas.
halos_sim = data_sim[(data_sim.pid == -1) & (data_sim.M200b > 0) &
                     np.isfinite(data_sim.cv)].copy()
print(f"SIM: {len(halos_sim)} halos anfitriones")

In [ ]:
def load_fastpm(path):
    """Lee el catálogo FastPM y devuelve DataFrame[x, y, z, M200b].

    ADAPTA esto a tu formato real. Ejemplos habituales (descomenta el tuyo):
    """
    # --- parquet ---
    df = pd.read_parquet(path)
    # --- HDF5 (h5py) ---
    # import h5py
    # with h5py.File(path, "r") as f:
    #     pos  = f["Position"][:]           # (N, 3)
    #     mass = f["Mass"][:]               # (N,)
    # df = pd.DataFrame({"x": pos[:,0], "y": pos[:,1], "z": pos[:,2], "M200b": mass})
    # --- npy ---
    # arr = np.load(path)                   # columnas [x, y, z, mass]
    # df = pd.DataFrame(arr, columns=["x", "y", "z", "M200b"])

    # Normaliza nombres de columnas -> x, y, z, M200b
    ren = {"X": "x", "Y": "y", "Z": "z", "Mass": "M200b", "mass": "M200b", "M": "M200b"}
    df = df.rename(columns={k: v for k, v in ren.items() if k in df.columns})
    return df[["x", "y", "z", "M200b"]]


halos_fastpm = load_fastpm(FASTPM_PATH)
halos_fastpm = halos_fastpm[halos_fastpm.M200b > 0].reset_index(drop=True)
print(f"FASTPM: {len(halos_fastpm)} halos (solo x, y, z, M200b)")

In [ ]:
# Opcional: Calibración de masa FASTPM → SIM¶
# Las masas de FastPM son aproximadas. Antes de condicionar por masa las remapeamos sobre la función de masa de SIM (abundance matching monótono). 
#Se guarda en la columna M200b_cal, que será la que se use para binear/condicionar.
if CALIBRATE_MASS:
    halos_fastpm["M200b_cal"] = abundance_match_mass(
        halos_fastpm.M200b.to_numpy(), halos_sim.M200b.to_numpy())
    MASS_COL_FASTPM = "M200b_cal"
else:
    MASS_COL_FASTPM = "M200b"

print("columna de masa FASTPM usada:", MASS_COL_FASTPM)

In [ ]:
#Proxy de entorno (feature de condicionamiento) para SIM y FASTPM
#Se calcula env idéntico en las dos simulaciones (mismo ENV_RADIUS, cada una con su boxsize periódico). 
#Es el sustituto de TidalForce que usabamos en un proyecto anterior, creo es computable en FASTPM. 
#Es la variable de entrada x del modelo condicional.


halos_sim["env"] = local_environment(
    halos_sim[["x", "y", "z"]].to_numpy(), SIM_BOXSIZE)
halos_fastpm["env"] = local_environment(
    halos_fastpm[["x", "y", "z"]].to_numpy(), FASTPM_BOXSIZE)

print("SIM    env  [min, mediana, max]:",
      np.round([halos_sim.env.min(), halos_sim.env.median(), halos_sim.env.max()], 3))
print("FASTPM env  [min, mediana, max]:",
      np.round([halos_fastpm.env.min(), halos_fastpm.env.median(), halos_fastpm.env.max()], 3))

In [ ]:
#entrenabamos un modelo por bin de masa, condicionando en env dentro de cada bin. 
#Habría que ajustar los bordes al nuevo rango de masas de SIM.
# Bordes de bins en log10(M200b). Ajusta al rango físico de SIM.

BIN_EDGES = np.array([10.0, 10.8, 11.6, 12.6, np.log10(halos_sim.M200b.max())])
N_MASS_BINS = len(BIN_EDGES) - 1
for i in range(N_MASS_BINS):
    print(f"bin {i}: 10^{BIN_EDGES[i]:.3g} - 10^{BIN_EDGES[i+1]:.3g} M_sun/h")

In [ ]:
#Validación hold-out en SIM (donde SÍ hay verdad)
#Antes de aplicar el modelo a FASTPM, para validar donde hay verdad: partimos SIM en train/test, entrenamos en train, predecimos en test y comparamos las distribuciones predichas frente a las reales. 
#Si aquí el modelo reproduce bien las distribuciones, podemos confiar en la transferencia a FASTPM.

rng = np.random.default_rng(0)
mask_test = rng.random(len(halos_sim)) < 0.2      # 20% test
sim_train = halos_sim[~mask_test]
sim_test  = halos_sim[mask_test]

val_results = []   # (i, y_true_test, y_pred_test)
for i in range(N_MASS_BINS):
    lo, hi = BIN_EDGES[i], BIN_EDGES[i + 1]
    tr = sim_train[(np.log10(sim_train.M200b) >= lo) & (np.log10(sim_train.M200b) < hi)]
    te = sim_test[(np.log10(sim_test.M200b)  >= lo) & (np.log10(sim_test.M200b)  < hi)]
    if len(tr) < 10 or len(te) < 10:
        val_results.append((i, None, None)); continue
    cg = ConditionalMultiVariateGaussian()
    cg.fit(tr[INPUT_FEATURES].to_numpy(), tr[OUTPUT_FEATURES].to_numpy())
    y_pred = cg.predict(te[INPUT_FEATURES].to_numpy())
    val_results.append((i, te[OUTPUT_FEATURES].to_numpy(), y_pred))
    print(f"bin {i}: train={len(tr)}, test={len(te)}  -> validado")

In [ ]:
#Entrenar en SIM completo y generar las propiedades en FASTPM
#Para cada bin de masa: entrenamos con todo SIM y predecimos las propiedades de los halos de FASTPM de ese bin (usando su env). 
#Las predicciones se van escribiendo en las nuevas columnas de halos_fastpm.
# Inicializa las columnas nuevas (a generar) en FASTPM.

for col in OUTPUT_FEATURES:
    halos_fastpm[col] = np.nan

logM_fastpm = np.log10(halos_fastpm[MASS_COL_FASTPM].to_numpy())
models = {}

for i in range(N_MASS_BINS):
    lo, hi = BIN_EDGES[i], BIN_EDGES[i + 1]

    sim_bin = halos_sim[(np.log10(halos_sim.M200b) >= lo) &
                        (np.log10(halos_sim.M200b) < hi)]
    fp_mask = (logM_fastpm >= lo) & (logM_fastpm < hi)
    print(f"bin {i}: SIM={len(sim_bin)}  FASTPM={fp_mask.sum()}")
    if len(sim_bin) < 10 or fp_mask.sum() == 0:
        continue

    cg = ConditionalMultiVariateGaussian()
    cg.fit(sim_bin[INPUT_FEATURES].to_numpy(), sim_bin[OUTPUT_FEATURES].to_numpy())
    y_pred = cg.predict(halos_fastpm.loc[fp_mask, INPUT_FEATURES].to_numpy())

    for j, col in enumerate(OUTPUT_FEATURES):
        halos_fastpm.loc[fp_mask, col] = y_pred[:, j]
    models[i] = cg

print("\nFASTPM enriquecido. Columnas:", list(halos_fastpm.columns))

In [ ]:
#Guardar el catálogo FASTPM aumentado

OUT = "FASTPM_augmented_z0.parquet"
halos_fastpm.to_parquet(OUT, index=False)
print("guardado:", OUT)
halos_fastpm.head()

In [ ]:
#Diagnósticos
#Validación en SIM: distribuciones reales (verdad) vs predichas en el test.
#Transferencia a FASTPM: distribuciones de SIM (verdad) vs FASTPM (generado). Como FASTPM no tiene verdad, esta comparación indica si el modelo reproduce la estadística esperada; las diferencias reflejan distintas distribuciones de masa/entorno entre SIM y FASTPM.

# Paletas y rangos (API moderna de matplotlib).
_ora = matplotlib.colormaps["Oranges"]
_bin = matplotlib.colormaps["binary"]
coloro  = [_ora(0.3), _ora(0.5), _ora(0.7), _ora(0.8)]
coloroo = [_ora(0.3), _ora(0.5), _ora(0.7), _ora(0.89)]
colorb  = [_bin(0.6)] * 4
RANGES = dict(ca=[0.25, 1.05], conc=[0.1, 30], ba=[0.45, 1.05], spin=[0, 0.1])
BINS_2D = 15


def compare_2d(ax, x_true, y_true, x_pred, y_pred_, rx, ry, alpha=0.26):
    """Contornos de confianza: verdad (relleno naranja) vs predicción (línea negra)."""
    edges = np.histogram2d(x_true, y_true, bins=BINS_2D, range=[rx, ry])
    ex, ey = bin_midpoints(edges[1]), bin_midpoints(edges[2])
    vt = edges[0].T / edges[0].sum()
    lvl = confidence_intervals(vt)
    ax.contourf(ex, ey, vt, levels=lvl, colors=coloroo, alpha=alpha)
    h_true = ax.contour(ex, ey, vt, levels=lvl, colors=coloro, linewidths=3).legend_elements()[0]
    vp = np.histogram2d(x_pred, y_pred_, bins=BINS_2D, range=[rx, ry])[0]
    vp = vp.T / vp.sum()
    lvl = confidence_intervals(vp)
    h_pred = ax.contour(ex, ey, vp, levels=lvl, colors=colorb,
                        linewidths=4, linestyles="dashed").legend_elements()[0]
    ax.tick_params(axis="both", direction="in")
    ax.yaxis.set_ticks_position("both"); ax.xaxis.set_ticks_position("both")
    return h_true, h_pred


def corner_plot(x_true_df, y_pred_arr, title, fname):
    """Figura corner 3x3 comparando verdad (df) vs predicción (array [cv,Spin,ca,ba])."""
    C = {c: k for k, c in enumerate(OUTPUT_FEATURES)}  # cv=0, Spin=1, ca=2, ba=3
    P = lambda c: y_pred_arr[:, C[c]]
    fig, axs = plt.subplots(3, 3, figsize=(15, 15),
                            gridspec_kw={"wspace": 0, "hspace": 0},
                            sharex="col", sharey="row")
    for a in (axs[0, 1], axs[0, 2]):
        a.axis("off")
    axs[1, 2].set_visible(False)

    ht, hp = compare_2d(axs[0, 0], x_true_df.Spin, x_true_df.cv, P("Spin"), P("cv"),
                        RANGES["spin"], RANGES["conc"])
    axs[0, 0].set_ylabel("Halo Concentration", fontsize=20)
    compare_2d(axs[1, 0], x_true_df.Spin, x_true_df.ca, P("Spin"), P("ca"),
               RANGES["spin"], RANGES["ca"]); axs[1, 0].set_ylabel("Halo Shape c/a", fontsize=20)
    compare_2d(axs[2, 0], x_true_df.Spin, x_true_df.ba, P("Spin"), P("ba"),
               RANGES["spin"], RANGES["ba"]); axs[2, 0].set_ylabel("Halo Shape b/a", fontsize=20)
    axs[2, 0].set_xlabel("Halo Spin", fontsize=20)
    compare_2d(axs[1, 1], x_true_df.cv, x_true_df.ca, P("cv"), P("ca"),
               RANGES["conc"], RANGES["ca"])
    compare_2d(axs[2, 1], x_true_df.cv, x_true_df.ba, P("cv"), P("ba"),
               RANGES["conc"], RANGES["ba"]); axs[2, 1].set_xlabel("Halo Concentration", fontsize=20)
    ax = axs[2, 2]
    xline = np.linspace(0, 1.1, 4)
    ax.fill_between(xline, xline, where=(xline > 0), color="grey", alpha=0.1)
    ax.set_xlim(0.3, 1.05); ax.set_ylim(0.38, 1.05)
    compare_2d(ax, x_true_df.ca, x_true_df.ba, P("ca"), P("ba"),
               RANGES["ca"], RANGES["ba"]); axs[2, 2].set_xlabel("Halo Shape c/a", fontsize=20)
    axs[0, 0].legend([ht[-2], hp[-2]], ["SIM (verdad)", "Predicción"],
                     fontsize=25, bbox_to_anchor=(3.05, 1))
    fig.suptitle(title, fontsize=22)
    plt.savefig(fname); plt.show()

In [ ]:
#Validación en SIM (test vs predicción)
for i, y_true, y_pred in val_results:
    if y_true is None:
        continue
    df_true = pd.DataFrame(y_true, columns=OUTPUT_FEATURES)
    corner_plot(df_true, y_pred, f"Validación SIM — bin de masa {i}",
                f"val_SIM_bin{i}.pdf")

In [ ]:
#Transferencia a FASTPM (SIM vs FASTPM generado)
for i in range(N_MASS_BINS):
    lo, hi = BIN_EDGES[i], BIN_EDGES[i + 1]
    sim_bin = halos_sim[(np.log10(halos_sim.M200b) >= lo) & (np.log10(halos_sim.M200b) < hi)]
    fp_mask = (logM_fastpm >= lo) & (logM_fastpm < hi)
    fp_bin  = halos_fastpm.loc[fp_mask & halos_fastpm[OUTPUT_FEATURES[0]].notna()]
    if len(sim_bin) < 10 or len(fp_bin) < 10:
        continue
    corner_plot(sim_bin, fp_bin[OUTPUT_FEATURES].to_numpy(),
                f"SIM vs FASTPM generado — bin de masa {i}", f"SIMvsFASTPM_bin{i}.pdf")

In [ ]:
#Relaciones propiedad–masa (opcional)
#Comprobación rápida de que la relación mediana propiedad–masa generada en FASTPM sigue la de SIM

prop = "cv"
rng_m = (halos_sim.M200b.min(), halos_sim.M200b.max())
def med_vs_mass(mass, val, mass_range):
    logM = np.log10(mass)
    x = 10**stats.binned_statistic(logM, logM, "mean",   bins=11, range=mass_range)[0]
    y = 10**stats.binned_statistic(logM, np.log10(val),  "median", bins=11, range=mass_range)[0]
    return x, y

xs, ys = med_vs_mass(halos_sim.M200b, halos_sim[prop], rng_m)
fp = halos_fastpm[halos_fastpm[prop].notna()]
xf, yf = med_vs_mass(fp[MASS_COL_FASTPM], fp[prop], rng_m)
plt.figure(figsize=(8, 6), dpi=80)
plt.plot(xs, ys, "-", label="SIM (verdad)")
plt.plot(xf, yf, "--", label="FASTPM (generado)")
plt.xscale("log"); plt.xlabel(r"M$_{200b}$"); plt.ylabel(f"mediana {prop}")
plt.legend(); plt.show()